# PicFlow feature exploration

This notebook mirrors the RGB histogram pipeline used by the PicFlow backend (`backend/app/ml/features.py` and `projection.py`):

1. Index RGB images from a folder
2. Build normalized per-channel histogram features
3. Cluster with k-means and visualize with PCA / UMAP
4. Optionally copy images into `output/` by cluster label (gitignored)

Run with the kernel working directory set to this `notebooks/` folder (e.g. `cd notebooks && jupyter lab`).


In [ ]:
import sys
from pathlib import Path

# PicFlow backend modules (same code the API uses for features and projections)
ROOT = Path.cwd().resolve()
if not (ROOT / "backend" / "app").is_dir() and (ROOT.parent / "backend" / "app").is_dir():
    ROOT = ROOT.parent
BACKEND = ROOT / "backend"
if str(BACKEND) not in sys.path:
    sys.path.insert(0, str(BACKEND))


In [ ]:
import shutil
import shelve

import matplotlib.pyplot as plt
import numpy as np
from tqdm.notebook import tqdm

from app.ml.features import extract_rgb_histogram, is_rgb_image_path
from app.ml.projection import compute_kmeans, compute_pca, compute_umap

NOTEBOOKS_DIR = ROOT / "notebooks" if (ROOT / "notebooks").is_dir() else ROOT
OUTPUT_ROOT = NOTEBOOKS_DIR / "output"
CACHE_PATH = NOTEBOOKS_DIR / "histogram_cache"
HISTOGRAM_BINS = 18


In [ ]:
# Absolute path to a folder of photographs on this machine
IMAGE_ROOT = Path.home() / ".PhotoChart" / "database" / "media" / "photographs"
MAX_IMAGES = 6000  # cap for interactive exploration; use None for all files


In [ ]:
def collect_rgb_paths(root: Path, limit: int | None = None) -> list[Path]:
    paths: list[Path] = []
    for path in sorted(root.rglob("*")):
        if not path.is_file():
            continue
        if not is_rgb_image_path(path):
            continue
        paths.append(path)
        if limit is not None and len(paths) >= limit:
            break
    return paths


def build_histogram_matrix(paths: list[Path], bins: int = HISTOGRAM_BINS):
    features: list[np.ndarray] = []
    kept: list[Path] = []
    for path in tqdm(paths, desc="Histograms"):
        hist = extract_rgb_histogram(path, histogram_bins=bins)
        if hist is None:
            continue
        features.append(hist)
        kept.append(path)
    return np.vstack(features), kept


In [ ]:
with shelve.open(str(CACHE_PATH)) as db:
    if "features_paths" in db:
        X, paths = db["features_paths"]
    else:
        all_paths = collect_rgb_paths(IMAGE_ROOT, limit=MAX_IMAGES)
        X, paths = build_histogram_matrix(all_paths)
        db["features_paths"] = X, paths

print(f"{len(paths)} RGB images, feature dim {X.shape[1]}")


In [ ]:
# Feature matrix X is already flattened (n_images, 3 * bins) from extract_rgb_histogram.
# Reshape for per-image RGB histogram plots.
hists = X.reshape(len(X), 3, HISTOGRAM_BINS)

n_plot = min(40, len(hists))
hists_plot = hists[:n_plot]
N, channels, bins = hists_plot.shape

fig = plt.figure(figsize=(12, 8))
ax = fig.add_subplot(111, projection="3d")
colors = ["red", "green", "blue"]
dx, dy = 0.8, 0.25
x = np.arange(bins)

for i in range(N):
    for c in range(channels):
        y = i + c * dy
        ax.bar3d(
            x,
            np.full(bins, y),
            np.zeros(bins),
            dx,
            dy,
            hists_plot[i, c, :],
            color=colors[c],
            alpha=0.8,
            shade=True,
        )

ax.set_xlabel("Histogram bin")
ax.set_ylabel("Image index")
ax.set_zlabel("Normalized mass")
ax.set_title("RGB histograms (first images)")
plt.tight_layout()
plt.show()


In [ ]:
n_clusters = 8
labels = compute_kmeans(X, n_clusters=n_clusters, random_state=42)
print("Cluster counts:", np.bincount(labels))


In [ ]:
coords_pca, explained = compute_pca(X, n_components=3)
print("Explained variance ratio:", [round(v, 4) for v in explained])

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection="3d")
scatter = ax.scatter(
    coords_pca[:, 0],
    coords_pca[:, 1],
    coords_pca[:, 2],
    c=labels,
    cmap="tab10",
    s=50,
    alpha=0.8,
)
ax.set_title(f"PCA of PicFlow histogram features ({n_clusters} clusters)")
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.set_zlabel("PC3")
fig.colorbar(scatter, ax=ax, label="Cluster", pad=0.1)
plt.tight_layout()
plt.show()


In [ ]:
coords_umap = compute_umap(X, n_components=3, random_state=42)

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection="3d")
scatter = ax.scatter(
    coords_umap[:, 0],
    coords_umap[:, 1],
    coords_umap[:, 2],
    c=labels,
    cmap="tab10",
    s=50,
    alpha=0.8,
)
ax.set_title(f"UMAP of PicFlow histogram features ({n_clusters} clusters)")
ax.set_xlabel("UMAP 1")
ax.set_ylabel("UMAP 2")
ax.set_zlabel("UMAP 3")
fig.colorbar(scatter, ax=ax, label="Cluster", pad=0.1)
plt.tight_layout()
plt.show()


In [ ]:
OUTPUT = OUTPUT_ROOT / f"KMeans_{n_clusters}"
OUTPUT.mkdir(parents=True, exist_ok=True)

for original, label in tqdm(zip(paths, labels), total=len(paths), desc="Export"):
    label_dir = OUTPUT / str(int(label))
    label_dir.mkdir(parents=True, exist_ok=True)
    dest = label_dir / original.name
    if not dest.exists():
        shutil.copy2(original, dest)

print("Wrote clusters under", OUTPUT)
